In [1]:
import matplotlib.pyplot as plt                 # static plots: trajectories, energy, μ, histograms
from matplotlib.animation import FuncAnimation  # animated orbits (save as .gif/.mp4)
from mpl_toolkits.mplot3d import Axes3D         # 3D axes (older Matplotlib versions only)

In [2]:
"""
pusher.py — the Boris particle pusher and a simple integration loop.
"""
import numpy as np                              

def boris_step(x, v, q_m, E, B, dt):
    """Advance one particle by one time step with the Boris scheme.

    x, v : position and velocity, shape (3,)
    q_m  : charge-to-mass ratio q/m
    E, B : field vectors at the particle, shape (3,)
    dt   : time step
    """
    # 1. half electric kick
    v_minus = v + q_m * E * dt / 2

    # 2. magnetic rotation (keeps |v| exactly constant)
    t = q_m * B * dt / 2
    s = 2 * t / (1 + t @ t)
    v_prime = v_minus + np.cross(v_minus, t)
    v_plus = v_minus + np.cross(v_prime, s)

    # 3. second half electric kick
    v_new = v_plus + q_m * E * dt / 2

    # 4. move the particle
    x_new = x + v_new * dt
    return x_new, v_new


def run(x0, v0, q_m, E_func, B_func, dt, n_steps):
    """Integrate a trajectory. Returns arrays t, x, v of length n_steps+1."""
    x = np.empty((n_steps + 1, 3))
    v = np.empty((n_steps + 1, 3))
    t = np.arange(n_steps + 1) * dt
    x[0], v[0] = x0, v0

    for i in range(n_steps):
        E = E_func(x[i], t[i])
        B = B_func(x[i], t[i])
        x[i + 1], v[i + 1] = boris_step(x[i], v[i], q_m, E, B, dt)

    return t, x, v